In [1]:
import requests
import pandas as pd
import time
from collections import defaultdict
import concurrent
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm  # Importing tqdm for the progress bar
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import seaborn as sb
import statsmodels.api as sm
from scipy.stats import pearsonr
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal

In [2]:
api_key = 'eb2cec89a26c7449245d9379a4f9944e'
#api_key = '41e72f874714fb6cdc138372a347c03b'
base_url = 'https://api.elsevier.com/content/search/scopus'
headers = {'Accept': 'application/json'}
query = 'TITLE-ABS-KEY("anesthesiology")'

# Function to fetch results with pagination using cursor
def fetch_results(cursor=None, year=None):
    params = {
        'query': query,
        'count': 25,  # Max number of results per page
        'date': year,
        'apiKey': api_key
    }
    
    # If we have a cursor, use it to fetch the next page
    if cursor:
        params['start'] = cursor
    
    response = requests.get(base_url, headers=headers, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching data for {year}: ", response.status_code)
        return None

    
 
# Function to collect unique country affiliations using cursor-based pagination
def extract_countries_from_entries(entries, country_counts):
    countries = set()
    for entry in entries:
        affiliations = entry.get('affiliation', [])
        for affil in affiliations:
            country = affil.get('affiliation-country', None)
            if country:
                # Increment the count for the country in the dictionary
                if country not in country_counts:
                    country_counts[country] = 0
                country_counts[country] += 1
            else:
                # If no country is provided, categorize as 'No Country'
                if 'No Country' not in country_counts:
                    country_counts['No Country'] = 0
                country_counts['No Country'] += 1
    return country_counts



# Function to process a page and get the next cursor
def process_page(cursor, country_counts, year):
    data = fetch_results(cursor, year)
    
    if not data:
        print(f"Error: No data received from the API for {year}.")
        return None, country_counts
    
    # Log the data for debugging if 'search-results' is missing
    if 'search-results' not in data:
        print(f"Error: 'search-results' not found in response for {year}. Full response:", data)
        return None, country_counts
    
    # Extract country data from current page
    country_counts = extract_countries_from_entries(data['search-results'].get('entry', []), country_counts)
    
    # Check for a 'next' link to get the cursor for the next page
    next_link = None
    for link in data.get('search-results', {}).get('link', []):
        if link['@ref'] == 'next':
            next_link = link['@href']
            break
    
    # If no next page, return None
    next_cursor = None
    if next_link:
        # Extract the 'start' parameter from the next page URL
        parsed_url = urlparse(next_link)
        next_cursor = parse_qs(parsed_url.query).get('start', [None])[0]
            
    return next_cursor, country_counts



def total(year):
    params = {
        'query': query,
        'count': 25,  # Max number of results per page
        'date': year,
        'apiKey': api_key
    }
    response = requests.get(base_url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        # Extract the total results from the JSON response
        total = data.get('search-results', {}).get('opensearch:totalResults', 0)
        return total
    else:
        print(f"Error fetching total results for {year}: ", response.status_code)
        return None



# Function to collect unique country affiliations using cursor-based pagination
def collect_articles_per_country(year):
    country_counts = {}  # Dictionary to store the count of articles per country
    cursor = 0  # Starting cursor position for the first request
    total_results = None  # Placeholder for total results count (used to estimate total pages)
    
    # Initialize the progress bar with no total (we will update it dynamically)
    with tqdm(desc=f"Processing Pages for {year}", unit="page") as pbar:
        
        # Using ThreadPoolExecutor to parallelize API requests
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            futures = []
            page_count = 0
            
            while cursor is not None:
                # Submit the next page request
                futures.append(executor.submit(process_page, cursor, country_counts, year))
                
                # Wait for the current batch to complete and collect results
                for future in concurrent.futures.as_completed(futures):
                    next_cursor, country_counts = future.result()
                    
                    # Move to the next page (if any)
                    cursor = next_cursor
                    
                    # If it's the first page, estimate the total number of pages
                    if total_results is None:
                        # Total results are available in the first response
                        data = future.result()[0]  # Get the data from the first page result
                        total_results = int(total(year))
                        # Estimate the number of pages
                        estimated_total_pages = (total_results // 25) + 1
                        pbar.total = estimated_total_pages  # Set total in progress bar
                    
                    # Update progress bar for every page processed
                    pbar.update(1)
                    
                    # Clear the futures list to prevent memory overload
                    futures = []
                    
                    # Be mindful of rate limits, sleep if necessary
                    time.sleep(0.1)  # Short sleep time to prevent hitting rate limits

    return country_counts

In [3]:
# Fetch all articles and their counts per country
years = range(1920,2022)
all_country_counts = defaultdict(int)

for year in years:
    print(f"Processing data for {year}...")
    country_counts = collect_articles_per_country(year)
    
    # Accumulate country counts across years
    for country, count in country_counts.items():
        all_country_counts[country] += count

Processing data for 1920...


Processing Pages for 1920: 100%|██████████| 1/1 [00:01<00:00,  1.03s/page]


Processing data for 1921...


Processing Pages for 1921: 100%|██████████| 1/1 [00:00<00:00,  1.21page/s]


Processing data for 1922...


...

Processing Pages for 2019: 100%|██████████| 61/61 [00:53<00:00,  1.14page/s]


Processing data for 2020...


Processing Pages for 2020: 100%|██████████| 65/65 [00:55<00:00,  1.17page/s]


Processing data for 2021...


Processing Pages for 2021: 100%|██████████| 67/67 [01:02<00:00,  1.08page/s]


In [10]:
#Dataframe Management after API Retrieval
country_df = pd.DataFrame(list(all_country_counts.items()), columns=['country', 'Article Count'])
country_df_dict = country_df['country']
country_df_dict.to_csv('affiliation_dictionary.csv', index=False)